# HYZ 2026 Public Colab

Private `HYZ_2026` ana reposunu submodule'lar ile klonlar, Task 1 `.pt` ağırlığını doğrular ve Task 2'nin gerçek yarışma akışını çalıştırır. Task 1 ve Task 3 bu sürümde bilinçli olarak boş payload üretir.


## 1. Colab Secrets

Şunları ekleyin:

- `GITHUB_TOKEN`
- `TEAM_NAME`
- `PASSWORD`
- `EVALUATION_SERVER_URL`
- `TASK1_MODEL_URL` — doğrudan indirilebilir `.pt` ağırlık bağlantısı

İsteğe bağlı: `TASK1_MODEL_SHA256` ile indirilen ağırlığın SHA-256 özeti doğrulanır. Secret değerleri ekrana basılmaz; `GITHUB_TOKEN` yalnızca GitHub clone işlemi ve GitHub'dan indirilen model bağlantısı için kullanılır.


In [ ]:
import os
import shutil
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

REPOSITORY_URL = "https://github.com/RaidynTeam/HYZ_2026.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_DIR = Path("/content/HYZ_2026")

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secrets içinde GITHUB_TOKEN eksik")

os.chdir("/content")
shutil.rmtree(REPOSITORY_DIR, ignore_errors=True)
askpass = Path("/content/.hyz_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
clone_environment = os.environ.copy()
clone_environment.update({
    "GITHUB_TOKEN": github_token,
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
})
try:
    subprocess.run(
        [
            "git", "clone", "--recurse-submodules",
            "--branch", REPOSITORY_BRANCH, "--single-branch",
            REPOSITORY_URL, str(REPOSITORY_DIR),
        ],
        check=True,
        env=clone_environment,
    )
    subprocess.run(
        ["git", "submodule", "update", "--init", "--recursive"],
        check=True,
        cwd=REPOSITORY_DIR,
        env=clone_environment,
    )
finally:
    askpass.unlink(missing_ok=True)
    clone_environment.pop("GITHUB_TOKEN", None)
    del github_token

os.chdir(REPOSITORY_DIR)
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
submodules = subprocess.check_output(
    ["git", "submodule", "status"], text=True
).strip()
print(f"Repository hazır: branch={REPOSITORY_BRANCH} commit={commit}")
print(submodules)


## 2. Task 1 `.pt` ağırlığını indir

Ağırlık Git'e girmez; Colab oturumunda `/content/models/task1.pt` olarak tutulur. Şimdilik sadece yüklenebilirliği doğrulanır, Task 1 payload'ı yine boş kalır.


In [ ]:
from hashlib import sha256
from urllib.parse import urlparse
from urllib.request import Request, urlopen

TASK1_MODEL_PATH = Path("/content/models/task1.pt")
task1_model_url = str(userdata.get("TASK1_MODEL_URL") or "").strip()
if not task1_model_url:
    raise RuntimeError("Colab Secrets içinde TASK1_MODEL_URL eksik")

github_hosts = {"github.com", "raw.githubusercontent.com", "api.github.com"}
headers = {}
github_token = None
if urlparse(task1_model_url).hostname in github_hosts:
    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("GitHub model bağlantısı için GITHUB_TOKEN eksik")
    headers["Authorization"] = f"Bearer {github_token}"

TASK1_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_model_path = TASK1_MODEL_PATH.with_suffix(".pt.part")
digest = sha256()
request = Request(task1_model_url, headers=headers)
with urlopen(request, timeout=180) as response, temporary_model_path.open("wb") as output:
    while chunk := response.read(1024 * 1024):
        output.write(chunk)
        digest.update(chunk)
if temporary_model_path.stat().st_size == 0:
    temporary_model_path.unlink(missing_ok=True)
    raise RuntimeError("Task 1 model dosyası boş indi")
expected_digest = str(userdata.get("TASK1_MODEL_SHA256") or "").strip().lower()
if expected_digest and digest.hexdigest() != expected_digest:
    temporary_model_path.unlink(missing_ok=True)
    raise RuntimeError("TASK1_MODEL_SHA256 uyuşmuyor; model kullanılmadı")
temporary_model_path.replace(TASK1_MODEL_PATH)
os.environ["TASK1_MODEL_PATH"] = str(TASK1_MODEL_PATH)
del task1_model_url, headers, request, github_token
print(f"Task 1 ağırlığı hazır: {TASK1_MODEL_PATH.name} ({TASK1_MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")


## 3. Python 3.12 ve bağımlılıklar

Ana repo/submodule bağımlılıkları kilitli dosyadan kurulur. Ardından yalnızca `.pt` açılış kontrolü için Ultralytics eklenir.


In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(
    ["uv", "sync", "--python", "3.12", "--frozen", "--extra", "dev"],
    cwd=REPOSITORY_DIR,
    check=True,
)
subprocess.run(
    ["uv", "pip", "install", "--python", ".venv/bin/python", "ultralytics>=8.3,<9"],
    cwd=REPOSITORY_DIR,
    check=True,
)
print("Python 3.12 ortamı, HYZ paketleri ve Task 1 model bağımlılığı hazır.")


## 4. Task 1 model preflight

Bu hücre yalnızca `.pt` dosyasının Ultralytics tarafından açılabildiğini kontrol eder. İnference çalıştırmaz ve Task 1 sonuçlarını değiştirmez.


In [ ]:
model_probe = """
from pathlib import Path
from ultralytics import YOLO

model_path = Path('/content/models/task1.pt')
if not model_path.is_file():
    raise FileNotFoundError(f'Task 1 model missing: {model_path}')
model = YOLO(str(model_path))
print(f'Task 1 .pt preflight passed: {model_path.name}')
"""
subprocess.run(
    [str(REPOSITORY_DIR / ".venv" / "bin" / "python"), "-c", model_probe],
    cwd=REPOSITORY_DIR,
    check=True,
)
print("Task 1 modeli hazır; yarışma payload'ı bu sürümde kasıtlı olarak boş kalacak.")


## 5. Yarışma ayarları

Secret'lar ignored `config/.env` dosyasına yazılır.


In [ ]:
def required_secret(name):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Colab Secrets içinde {name} eksik")
    return str(value)

def quote_env(value):
    clean = value.replace("\n", "").replace("\r", "")
    return '"' + clean.replace("\\", "\\\\").replace('\"', '\\"') + '"'

environment_values = {
    "TEAM_NAME": required_secret("TEAM_NAME"),
    "PASSWORD": required_secret("PASSWORD"),
    "EVALUATION_SERVER_URL": required_secret("EVALUATION_SERVER_URL").rstrip("/") + "/",
    "SESSION_NAME": "",
}
environment_path = REPOSITORY_DIR / "config" / ".env"
environment_path.write_text(
    "".join(f"{key}={quote_env(value)}\n" for key, value in environment_values.items()),
    encoding="utf-8",
)
environment_path.chmod(0o600)
del environment_values
print("Private config/.env hazır; secret değerleri gösterilmedi. Task 1 ve Task 3 boş payload modunda.")


## 6. Doğrulama

Server'a bağlanmaz; testleri çalıştırır.


In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run(
        ["uv", "run", "--frozen", "--extra", "dev", "python", "-m", "pytest", "-q"],
        cwd=REPOSITORY_DIR,
        check=True,
    )
else:
    print("Testler kullanıcı tarafından atlandı.")


## 7. Yarışmayı başlat

`START_COMPETITION = True` olunca gerçek server'a gönderim başlar.


In [ ]:
START_COMPETITION = False

if START_COMPETITION:
    model_path = Path("/content/models/task1.pt")
    if not model_path.is_file():
        raise RuntimeError("Task 1 .pt modeli bulunamadı; başlatma iptal edildi")
    print("Gönderim modu: Task 2 gerçek trajectory | Task 1 boş | Task 3 boş")
    subprocess.run(
        ["uv", "run", "--frozen", "python", "-u", "main.py"],
        cwd=REPOSITORY_DIR,
        check=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
else:
    print("Güvenli bekleme: START_COMPETITION=False, server'a prediction gönderilmedi.")
